[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%208/L16_Agentic_Support_Copilot.ipynb)

# Agentic Support Copilot
### ISBA 2411 · Week 8 · Lecture 16

**What you are building.** A support agent that reads an incoming customer ticket, searches
your company's own documentation, and drafts a reply in which every factual claim carries a
citation back to the passage it came from. It judges its own retrieval: when it is not
confident it widens the search and tries again, and when the answer genuinely is not in the
documentation it declines rather than inventing one. It retrieves across screenshots as well
as text, and it ships as a browser app a support rep can open.

Last lecture you watched a version of this answer a customer and then refuse twice. Tonight you
build it, and you add the two things it could not do: **choose its own steps**, and **read a
screenshot**.

> **This one is a follow-along.** Run every cell. Run the setup cell now, before the lecture
> starts, because it downloads about 3 GB.

Six blocks:

| block | what you build | topic |
|---|---|---|
| 1 | RAG in about forty lines, no framework | RAG |
| 2 | The same answer, made unpredictable | NLG |
| 3 | The same pipeline again, in LangChain | LangChain |
| 4 | Retrieval across images and text | Multimodal LMs |
| 5 | A graph that decides to search again | LangGraph, agents |
| 6 | A browser interface anyone can open | Streamlit |

**Runtime > Change runtime type > T4 GPU.** It works on CPU but the generation steps
take a few minutes each instead of a few seconds.

---
## Setup

Run this first. It is the only slow cell.

In [ ]:
%%capture
%pip install -q langchain langchain-community langchain-huggingface langgraph grandalf \
                sentence-transformers transformers accelerate streamlit

---
# Block 1 · RAG with no framework

The claim from last lecture was that a first RAG system is about forty lines. Here are the
forty lines. Nothing in this block imports LangChain.

#### ▶ STEP 1 &middot; Load Cobalt's help centre

In [ ]:
# -------- STEP 1 · Load Cobalt's help centre --------
import json, urllib.request, textwrap, warnings, numpy as np, torch
import transformers

# these libraries are chatty and the warnings are not about your code
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
kb = json.loads(urllib.request.urlopen("https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/data/cobalt_kb.json").read())

print(f"{len(kb)} chunks from {len(set(d['doc_id'] for d in kb))} help articles")
print(f"chunk length: {min(d['words'] for d in kb)} to {max(d['words'] for d in kb)} words")
for d in kb[:5]:
    print(f"   {d['title'][:32]:34} / {d['section']}")

if DEVICE == "cuda":
    print(f"\nrunning on {DEVICE}. The generator will load in float16, about 3.1 GB.")
else:
    print("\n" + "!" * 78)
    print("NO GPU. Everything below still works, but the generator loads in float32,")
    print("which is 6.2 GB instead of 3.1 GB, and each generation takes minutes not seconds.")
    print("Fix it now: Runtime > Change runtime type > T4 GPU, then re-run from this cell.")
    print("!" * 78)

Two of the eight ticket categories from Lecture 13 have **no article at all**. That is
deliberate: it means the system has real gaps to hit rather than staged ones, and you will
use one of them in Block 5.

#### ▶ STEP 2 &middot; Search it, with no framework

In [ ]:
# -------- STEP 2 · Search it, with no framework --------
from sentence_transformers import SentenceTransformer

enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

# index the heading alongside the body: the section title carries a lot of signal
X = enc.encode([f"{d['title']}. {d['section']}. {d['text']}" for d in kb],
               normalize_embeddings=True, batch_size=32)

def search(question, k=3):
    sims = X @ enc.encode([question], normalize_embeddings=True)[0]
    return [(kb[i], float(sims[i])) for i in np.argsort(sims)[::-1][:k]]

TICKET = "Our SSO through Okta stopped working after the weekend. Nobody can sign in."
for d, s in search(TICKET):
    print(f"  {s:.3f}  {d['title'][:30]:32} / {d['section']}")

That is the retrieval half, and it is the encoder you built in Lecture 13. **The index is
`X`: one row per chunk, 384 numbers each.** Search is one matrix multiply.

#### ▶ STEP 3 &middot; Answer from it, with no framework

In [ ]:
# -------- STEP 3 · Answer from it, with no framework --------
from transformers import AutoTokenizer, AutoModelForCausalLM

GEN = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(GEN)
gen = AutoModelForCausalLM.from_pretrained(
        GEN, dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE).eval()

SYSTEM = ("You are a Cobalt support agent. You may ONLY use facts from the numbered passages.\n"
          "After EVERY sentence that states a fact, put the passage number in brackets, e.g. [2].\n"
          "If the passages do not answer the ticket, reply with exactly NO_ANSWER and nothing else.\n"
          "Example: Go to Admin, then Identity Providers, and choose Reconnect [1].")

def answer(question, k=3, **kw):
    hits = search(question, k)
    ctx = "\n".join(f"[{n}] ({d['title']} / {d['section']}) {d['text']}"
                    for n, (d, _) in enumerate(hits, 1))
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Passages:\n{ctx}\n\nTicket:\n{question}"}]
    ids = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True),
              return_tensors="pt").to(DEVICE)
    kw = {"max_new_tokens": 160, "do_sample": False, "pad_token_id": tok.eos_token_id, **kw}
    with torch.no_grad():
        out = gen.generate(**ids, **kw)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip(), hits

reply, hits = answer(TICKET)
print(textwrap.fill(reply, 92))

✅ **That is a complete RAG system.** Chunk, embed, search, prompt. No vector database, no
framework, no training. Every `[n]` points at a passage you can read.

💼 **At work this means:** you can write this in an afternoon and you will understand every
failure it has, because there is nothing between you and it.

---
# Block 2 · NLG: one setting decides whether you can audit it

The model does not output a word. At every position it outputs a probability distribution
over the whole vocabulary. **Decoding is the rule you use to pick from that distribution.**

#### ▶ STEP 4 &middot; One setting: greedy against sampling

In [ ]:
# -------- STEP 4 · One setting: greedy against sampling --------
print("GREEDY, three runs. Same input, same output every time.\n")
for i in range(3):
    r, _ = answer(TICKET)
    print(f"  run {i+1}: {r[:96]}")

print("\nSAMPLING, three runs. Same input, different output every time.\n")
for i in range(3):
    r, _ = answer(TICKET, do_sample=True, temperature=0.9, top_p=0.92)
    print(f"  run {i+1}: {r[:96]}")

✅ **Look at what changed.** Greedy takes the single most likely next word every time, so the
output is reproducible. Sampling draws from the distribution, so two customers asking the
same question get different answers.

💼 **At work this means:** support answers run greedy. If you cannot reproduce an answer you
cannot write a regression test for it, you cannot debug a complaint about it, and you will
have trouble defending it. Sampling is for drafting and marketing, where variety is the point.

---
# Block 3 · The same pipeline, in LangChain

Now rebuild exactly what you already have, using the framework. Watch how much code
disappears, and pay attention to what disappears **with** it.

#### ▶ STEP 5 &middot; The same retriever, in LangChain

In [ ]:
# -------- STEP 5 · The same retriever, in LangChain --------
import importlib.metadata as meta
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import pipeline

# The 1.x API differs from most tutorials you will find by searching. Print what you have,
# because "that example does not work" is almost always a version mismatch.
print(" · ".join(f"{p} {meta.version(p)}"
                 for p in ["langchain", "langchain-core", "langgraph", "transformers"]))

docs = [Document(page_content=f"{d['title']}. {d['section']}. {d['text']}",
                 metadata={"title": d["title"], "section": d["section"]}) for d in kb]

lc_emb  = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
store   = InMemoryVectorStore.from_documents(docs, lc_emb)
retriever = store.as_retriever(search_kwargs={"k": 3})

for d in retriever.invoke(TICKET):
    print(f"  {d.metadata['title'][:30]:32} / {d.metadata['section']}")

#### ▶ STEP 6 &middot; The same chain, and a bug the framework hides

In [ ]:
# -------- STEP 6 · The same chain, and a bug the framework hides --------
llm = HuggingFacePipeline(pipeline=pipeline(
        "text-generation", model=gen, tokenizer=tok,      # reuse the model already loaded
        max_new_tokens=160, do_sample=False, return_full_text=False))

prompt = ChatPromptTemplate.from_messages([("system", SYSTEM),
                                           ("human", "Passages:\n{context}\n\nTicket:\n{question}")])

def format_docs(ds):
    return "\n".join(f"[{n}] {d.page_content}" for n, d in enumerate(ds, 1))

# LCEL: the pipe operator composes the steps into one runnable
chain = ({"context": retriever | format_docs, "question": lambda q: q}
         | prompt | llm | StrOutputParser())

print("A.  chain built on HuggingFacePipeline\n")
print(textwrap.fill(chain.invoke(TICKET).strip()[:400], 92))

**Read that output before you go on.** Compare it to STEP 3, which used the same model, the
same passages and the same system prompt.

It has drifted into writing *both sides of the conversation*, it ignored the citation rule,
and it invented steps that are not in any passage. Nothing errored. Nothing warned you.

The cause: `HuggingFacePipeline` is a **completion** model, not a chat model. You handed
LangChain a `("system", ...)` message, it accepted it without complaint, and then flattened
the whole thing into one plain string, so the chat template that Qwen needs was never applied.
In STEP 3 you applied it yourself with `tok.apply_chat_template` and never had to think about it.

The fix is one wrapper.

#### ▶ STEP 7 &middot; The fix: one wrapper

In [ ]:
# -------- STEP 7 · The fix: one wrapper --------
from langchain_huggingface import ChatHuggingFace

# ChatHuggingFace knows the model is a chat model and applies its template
chat = ChatHuggingFace(llm=llm, tokenizer=tok)
chat_chain = ({"context": retriever | format_docs, "question": lambda q: q}
              | prompt | chat | StrOutputParser())

print("B.  the same chain, wrapped in ChatHuggingFace\n")
print(textwrap.fill(chat_chain.invoke(TICKET).strip(), 92))

✅ **What the framework bought you.** The vector store, the retriever interface, the prompt
template and LCEL composition are standard parts now. Swapping `InMemoryVectorStore` for
Pinecone or pgvector is close to a one-line change, and that is genuinely worth something.

⚠️ **What it cost you.** Two things, and you just hit both.

First, the chat-template bug above. The abstraction accepted an invalid combination and
produced plausible garbage rather than an error. You could only diagnose it because you had
STEP 3 to compare against.

Second, visibility. In Block 1 you could print `X`, print the similarity scores, and see
exactly which chunk won and by how much. Here `retriever | format_docs` is opaque, and
**most RAG failures are retrieval failures**, so you are now debugging through a layer.

💼 **At work this means:** write the forty lines first. Adopt the framework when you can name
the specific problem it solves for you, not because the tutorials start there.

---
# Block 4 · Multimodal: one space for screenshots and sentences

Roughly half of real support tickets arrive with an image attached, and everything above is
blind to all of them. **CLIP** trains an image encoder and a text encoder together so that a
photograph and a sentence describing it land near each other in the same space.

The four screenshots below are drawn here rather than downloaded, so you can see exactly what
is in them. They are synthetic, but the retrieval is real.

#### ▶ STEP 8 &middot; Draw four screenshots

In [ ]:
# -------- STEP 8 · Draw four screenshots --------
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
import io

def shot(draw, name):
    fig, ax = plt.subplots(figsize=(3.2, 2.2), dpi=90); draw(ax)
    ax.set_xticks([]); ax.set_yticks([])
    buf = io.BytesIO(); fig.savefig(buf, format="png", bbox_inches="tight"); plt.close(fig)
    return name, Image.open(buf).convert("RGB")

def broken(ax):
    ax.set_title("Revenue by region", fontsize=9)
    ax.text(.5, .5, "No data to display", ha="center", color="#B91C1C", fontsize=11)
def login(ax):
    ax.set_title("Sign in", fontsize=9)
    ax.text(.5, .6, "Authentication failed", ha="center", color="#B91C1C", fontsize=11)
    ax.text(.5, .38, "SAML assertion invalid", ha="center", color="#6B7280", fontsize=8)
def slow(ax):
    ax.set_title("Dashboard", fontsize=9)
    ax.text(.5, .5, "Loading...  47s", ha="center", color="#D97706", fontsize=11)
def export(ax):
    ax.set_title("Export to CSV", fontsize=9)
    ax.text(.5, .55, "Limit exceeded", ha="center", color="#B91C1C", fontsize=11)
    ax.text(.5, .35, "10,000 rows maximum", ha="center", color="#6B7280", fontsize=8)

SHOTS = [shot(*a) for a in [(broken,"broken chart"), (login,"login error"),
                            (slow,"slow dashboard"), (export,"export limit")]]

fig, axes = plt.subplots(1, 4, figsize=(13, 2.4))
for ax, (n, im) in zip(axes, SHOTS):
    ax.imshow(im); ax.set_title(n, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

#### ▶ STEP 9 &middot; Load CLIP and embed the images

In [ ]:
# -------- STEP 9 · Load CLIP and embed the images --------
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# transformers 4.x returns a plain tensor here; 5.x returns an output object. Colab could
# have either, so take whichever came back.
def _vec(r): return r if torch.is_tensor(r) else r.pooler_output

def clip_images(imgs):
    b = proc(images=imgs, return_tensors="pt").to(DEVICE)
    with torch.no_grad(): v = _vec(clip.get_image_features(**b))
    return torch.nn.functional.normalize(v, dim=-1)

def clip_text(txts):
    b = proc(text=txts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    with torch.no_grad(): v = _vec(clip.get_text_features(**b))
    return torch.nn.functional.normalize(v, dim=-1)

IV = clip_images([im for _, im in SHOTS])
print(f"four screenshots -> {tuple(IV.shape)}   (one row per image)")
print("the SAME shape of object as your text index X. That is the whole idea.")

#### ▶ STEP 10 &middot; Find a screenshot by describing it

In [ ]:
# -------- STEP 10 · Find a screenshot by describing it --------
QUERIES = ["the chart is not rendering any data",
           "I cannot sign in, it says my login is invalid",
           "the page takes forever to load",
           "my download stops part way through"]

TV = clip_text(QUERIES)
names = [n for n, _ in SHOTS]

print(f"{'what the customer typed':46} {'best match':18} score   runner-up\n" + "-"*92)
for q, row in zip(QUERIES, TV @ IV.T):
    order = row.argsort(descending=True)
    a, b = order[0].item(), order[1].item()
    print(f"{q[:44]:46} {names[a]:18} {row[a]:.3f}   {names[b]} ({row[b]:.3f})")

✅ **Nothing was trained and no captions were written.** CLIP was pretrained on image-text
pairs, so cosine similarity between a sentence and a picture is already a meaningful number.
Once images and text share a space, every operation from Lecture 13 works across both.

⚠️ **The fourth query is the honest one, and it is wrong.** "My download stops part way
through" should land on the export limit dialog. CLIP returns the slow dashboard instead,
because it has latched onto "stops part way" as a statement about slowness.

Look at the scores as well as the ranks. All four sit between roughly 0.27 and 0.30, which is
a narrow band. **A top-1 result separated from second place by 0.01 is not a confident
answer**, and this is exactly the case for a refusal threshold like the one in Block 5.

💼 **At work this means:** two patterns. Caption the screenshot and run your existing text
pipeline, which is cheap and keeps one system. Or send the image and the question to a
vision-language model, which is better on fine detail like the exact error string.

---
# Block 5 · LangGraph: a pipeline that decides what to do next

Everything so far runs a fixed path: retrieve once, generate once. A **graph** lets the system
look at what came back and decide whether that was good enough.

Start by drawing the fixed pipeline you already have as a graph, so the new part is obvious.

#### ▶ STEP 11 &middot; The fixed pipeline, drawn as a graph

In [ ]:
# -------- STEP 11 · The fixed pipeline, drawn as a graph --------
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    hits: List[dict]
    k: int
    tries: int
    reply: str

def n_retrieve(s):
    return {"hits": [d for d, _ in search(s["question"], s["k"])], "tries": s["tries"] + 1}

def n_answer(s):
    ctx = "\n".join(f"[{n}] ({d['title']} / {d['section']}) {d['text']}"
                    for n, d in enumerate(s["hits"], 1))
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Passages:\n{ctx}\n\nTicket:\n{s['question']}"}]
    ids = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True),
              return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = gen.generate(**ids, max_new_tokens=160, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    return {"reply": tok.decode(out[0][ids["input_ids"].shape[1]:],
                                skip_special_tokens=True).strip()}

g = StateGraph(State)
g.add_node("retrieve", n_retrieve); g.add_node("answer", n_answer)
g.add_edge(START, "retrieve"); g.add_edge("retrieve", "answer"); g.add_edge("answer", END)
straight = g.compile()

print(straight.get_graph().draw_ascii())

That is the same pipeline as Block 1, drawn as a graph. It cost you more code and bought you
nothing, which is the honest starting point. **Now add the piece that makes it worth it.**

#### ▶ STEP 12 &middot; Add a grader, so it can retry

In [ ]:
# -------- STEP 12 · Add a grader, so it can retry --------
# The grader is an ordinary function. It is allowed to say "that was not good enough".
def n_grade(s):
    best = search(s["question"], 1)[0][1]
    print(f"   [grade] try {s['tries']}, k={s['k']}, best similarity {best:.3f}")
    return {}

def decide(s):
    best = search(s["question"], 1)[0][1]
    if best >= 0.45:      return "answer"     # confident: answer from what we have
    if s["tries"] >= 3:   return "refuse"     # tried enough, the answer is not in the KB
    return "widen"                            # not confident: look at more of the corpus

def n_widen(s):
    print(f"   [widen] k {s['k']} -> {s['k'] + 3}, searching again")
    return {"k": s["k"] + 3}

def n_refuse(s):
    return {"reply": "NO_ANSWER"}

g = StateGraph(State)
for name, fn in [("retrieve", n_retrieve), ("grade", n_grade), ("widen", n_widen),
                 ("answer", n_answer), ("refuse", n_refuse)]:
    g.add_node(name, fn)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "grade")
g.add_conditional_edges("grade", decide,
                        {"answer": "answer", "widen": "widen", "refuse": "refuse"})
g.add_edge("widen", "retrieve")          # <-- the loop
g.add_edge("answer", END); g.add_edge("refuse", END)
agent = g.compile()

print(agent.get_graph().draw_ascii())

#### ▶ STEP 13 &middot; Watch it loop, and watch it refuse

In [ ]:
# -------- STEP 13 · Watch it loop, and watch it refuse --------
def run(question):
    print(f"TICKET  {question}\n")
    final = None
    for ev in agent.stream({"question": question, "hits": [], "k": 3, "tries": 0, "reply": ""}):
        for node, upd in ev.items():
            print(f"   [{node}]")
            if upd and upd.get("reply"): final = upd["reply"]
    print(f"\nREPLY   {textwrap.fill(final, 88) if final else '(none)'}\n" + "="*90 + "\n")

run("Our SSO through Okta stopped working after the weekend. Nobody can sign in.")
run("Please add dark mode. Our team works late and the white background is rough.")

✅ **Read the two traces.** The first ticket is answered on the first pass: retrieval was
confident, the grader let it through, one generation. The second ticket has **no article in
the help centre at all**, so the grader is never satisfied, the graph widens the search and
tries again, and eventually gives up and refuses.

**Nothing in the code said how many searches to run.** The number of steps depended on the
ticket. That is the entire difference between a pipeline and an agent.

⚠️ **And it is why agents are harder to operate.** Your cost per ticket is now unbounded at
design time. The `tries >= 3` cap is not a detail, it is the thing standing between you and a
runaway loop. In production you also want a token budget and an approval gate on any tool
that writes rather than reads.

💼 **At work this means:** start with the straight pipeline. Move to a graph when you can point
at tickets that genuinely need a different number of steps.

---
# Block 6 · Streamlit: put it in front of a person

A notebook is not a product. Streamlit turns a script into something a colleague can open in a
browser, which is how you get a real user in front of a prototype this week rather than next
quarter.

**The app runs the agent from Block 5, not the straight pipeline**, and it prints the trace as
it goes. A user can watch it retrieve, decide it is not confident, widen the search, and either
answer or give up. The three sliders in the sidebar are the grader's threshold, the try limit
and the starting `k`, which are the three numbers that decide how the agent behaves.

In [ ]:
%%writefile app.py
# The whole product in one file, running the AGENT from Block 5 rather than the straight
# pipeline. The trace panel shows every node as it fires, so a user can watch it decide to
# search again.
#
# NOTE: no bare `A if cond else B` statements anywhere below. Streamlit treats a bare
# expression as something to display and re-parses the source line to name it, which breaks
# on a line continuation. Use plain if/else in a Streamlit script.
import json, urllib.request, numpy as np, streamlit as st, torch
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

st.set_page_config(page_title="Cobalt Copilot", page_icon="🟦", layout="wide")
KB = "https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/data/cobalt_kb.json"
SYSTEM = ("You are a Cobalt support agent. You may ONLY use facts from the numbered passages.\n"
          "After EVERY sentence that states a fact, put the passage number in brackets, e.g. [2].\n"
          "If the passages do not answer the ticket, reply with exactly NO_ANSWER and nothing else.\n"
          "Example: Go to Admin, then Identity Providers, and choose Reconnect [1].")

@st.cache_resource(show_spinner=False)
def load():
    from sentence_transformers import SentenceTransformer
    from transformers import AutoTokenizer, AutoModelForCausalLM
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    with st.status("Starting the copilot…", expanded=True) as s:
        st.write(f"running on **{dev}**")
        kb = json.loads(urllib.request.urlopen(KB).read())
        st.write(f"indexing {len(kb)} chunks…")
        enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=dev)
        X = enc.encode([f"{d['title']}. {d['section']}. {d['text']}" for d in kb],
                       normalize_embeddings=True, batch_size=32)
        st.write("loading the generator, about 3 GB on a cold machine…")
        tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
        gen = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct",
                dtype=torch.float16 if dev == "cuda" else torch.float32).to(dev).eval()
        s.update(label=f"Ready. {len(kb)} chunks on {dev}.", state="complete", expanded=False)
    return kb, enc, X, tok, gen, dev

kb, enc, X, tok, gen, DEV = load()

def search(q, k):
    sims = X @ enc.encode([q], normalize_embeddings=True)[0]
    return [(kb[i], float(sims[i])) for i in np.argsort(sims)[::-1][:k]]

# ------------------------------------------------------------------ the agent
class State(TypedDict):
    question: str
    hits: List[dict]
    scores: List[float]
    k: int
    tries: int
    reply: str

def n_retrieve(s):
    hits = search(s["question"], s["k"])
    return {"hits": [d for d, _ in hits], "scores": [v for _, v in hits],
            "tries": s["tries"] + 1}

def n_grade(s):
    return {}

def n_widen(s):
    return {"k": s["k"] + 3}

def n_answer(s):
    ctx = "\n".join(f"[{n}] ({d['title']} / {d['section']}) {d['text']}"
                    for n, d in enumerate(s["hits"], 1))
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Passages:\n{ctx}\n\nTicket:\n{s['question']}"}]
    ids = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True),
              return_tensors="pt").to(DEV)
    with torch.no_grad():
        out = gen.generate(**ids, max_new_tokens=160, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    return {"reply": tok.decode(out[0][ids["input_ids"].shape[1]:],
                                skip_special_tokens=True).strip()}

def n_refuse(s):
    return {"reply": "NO_ANSWER"}

def build_agent(threshold, max_tries):
    def decide(s):
        best = s["scores"][0] if s["scores"] else 0.0
        if best >= threshold:      return "answer"
        if s["tries"] >= max_tries: return "refuse"
        return "widen"
    g = StateGraph(State)
    for name, fn in [("retrieve", n_retrieve), ("grade", n_grade), ("widen", n_widen),
                     ("answer", n_answer), ("refuse", n_refuse)]:
        g.add_node(name, fn)
    g.add_edge(START, "retrieve")
    g.add_edge("retrieve", "grade")
    g.add_conditional_edges("grade", decide,
                            {"answer": "answer", "widen": "widen", "refuse": "refuse"})
    g.add_edge("widen", "retrieve")
    g.add_edge("answer", END)
    g.add_edge("refuse", END)
    return g.compile()

# ----------------------------------------------------------------------- UI
st.title("Cobalt Support Copilot")
st.caption("Runs the LangGraph agent from Block 5. Watch the trace decide whether to "
           "search again.")

st.sidebar.title("The dials")
threshold = st.sidebar.slider("Answer when similarity reaches", 0.20, 0.70, 0.45, 0.01,
    help="Lower = answers more often, and is wrong more often.")
max_tries = st.sidebar.slider("Give up after this many tries", 1, 5, 3)
start_k   = st.sidebar.slider("Start by retrieving", 1, 6, 3)
st.sidebar.caption(f"{len(kb)} chunks · running on {DEV}")

EXAMPLES = {
 "SSO broken for the whole team":
   "Our SSO through Okta stopped working after the weekend. Nobody can sign in.",
 "→ Not in the documentation at all":
   "Please add dark mode. Our team works late and the white background is rough.",
 "Export times out":
   "Exporting anything over about 10,000 rows just times out. We need the full extract.",
}
if "ticket" not in st.session_state:
    st.session_state.ticket = EXAMPLES["SSO broken for the whole team"]

def _pick():
    st.session_state.ticket = EXAMPLES.get(st.session_state.example, "")

st.selectbox("Start from an example", list(EXAMPLES), key="example", on_change=_pick)
q = st.text_area("Customer ticket", key="ticket", height=100)
go = st.button("Draft a reply", type="primary", disabled=not q.strip())

if go:
    agent = build_agent(threshold, max_tries)
    init = {"question": q, "hits": [], "scores": [], "k": start_k, "tries": 0, "reply": ""}
    final, hits, scores = None, [], []

    with st.status("Running the agent…", expanded=True) as status:
        for ev in agent.stream(init):
            for node, upd in ev.items():
                if node == "retrieve":
                    hits, scores = upd["hits"], upd["scores"]
                    st.write(f"**retrieve** &middot; try {upd['tries']}, "
                             f"best similarity `{scores[0]:.3f}`")
                elif node == "widen":
                    st.write(f"**widen** &middot; not confident enough, "
                             f"retrieving {upd['k']} passages instead")
                elif node == "answer":
                    final = upd["reply"]
                    st.write("**answer** &middot; generating from the passages")
                elif node == "refuse":
                    final = upd["reply"]
                    st.write("**refuse** &middot; the answer is not in the help centre")
        status.update(label="Done.", state="complete", expanded=True)

    left, right = st.columns([1.05, 1])
    with left:
        st.subheader("The drafted reply")
        if final is None:
            st.info("The agent produced no reply.")
        elif final.upper().startswith("NO_ANSWER"):
            st.warning("**I do not have that in our documentation.** "
                       "This ticket is going to a human.")
        else:
            st.success(final)
            st.caption("Every [n] refers to a passage on the right. A claim with no number "
                       "was not grounded, and that is the thing to look for.")
    with right:
        st.subheader("What it retrieved")
        for n, (d, sc) in enumerate(zip(hits, scores), 1):
            with st.container(border=True):
                st.markdown(f"**[{n}] {d['title']}**  \n*{d['section']}*  &middot;  `{sc:.3f}`")
                st.caption(d["text"])


### Now restart the runtime

`app.py` is on disk. The app will load **its own** copy of the generator, and this notebook is
still holding one. Two copies do not fit in Colab: on a CPU runtime the model is float32, which
is 6.2 GB each, against about 12.7 GB of RAM.

Deleting the variables does not help, and it is worth understanding why. `gen` is also
referenced by the LangChain pipeline, by both chains, and by the closures inside your LangGraph
nodes. Python frees an object when the **last** reference goes, so `del gen` on its own frees
nothing at all.

Restarting is the reliable fix. Your installed packages and `app.py` both survive it.

#### ▶ STEP 14 &middot; Restart the runtime, so the app starts clean

In [ ]:
# -------- STEP 14 · Restart the runtime, so the app starts clean --------
# This kills the kernel ON PURPOSE.
#
# Colab will show a red banner saying the session crashed or the runtime disconnected.
# THAT IS THIS CELL WORKING. It is not an error and nothing is lost.
#
# When you see it: scroll to STEP 15 and run that one cell. Nothing else.
import os
print("Restarting on purpose. The red 'session crashed' banner is expected.")
print("When you see it, run STEP 15 and only STEP 15.")
os.kill(os.getpid(), 9)

#### ▶ STEP 15 &middot; AFTER THE RESTART: launch the app

In [ ]:
# -------- STEP 15 · AFTER THE RESTART: launch the app --------
# Run this AFTER the restart above. It needs nothing from the notebook: app.py is on disk
# and your pip installs survived.
# Colab cannot show a web server directly, so we open a public tunnel to it.
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
get_ipython().system_raw("streamlit run app.py --server.port 8501 --server.headless true &")
get_ipython().system_raw("./cloudflared tunnel --url http://localhost:8501 > tunnel.log 2>&1 &")

import re, time
for _ in range(30):
    time.sleep(2)
    try:
        m = re.search(r"https://[-\w]+\.trycloudflare\.com", open("tunnel.log").read())
        if m:
            print("\n   OPEN THIS:", m.group(0), "\n")
            print("   First load takes a minute while the app loads its own copy of the model.")
            break
    except FileNotFoundError:
        pass
else:
    print("tunnel did not come up. Run the cell again, or check tunnel.log")

✅ **That is the whole arc.** Forty lines of RAG, the same thing in a framework, a graph that
decides for itself, retrieval that crosses from text into images, and a browser interface.

⚠️ **Streamlit is a prototyping tool.** No authentication, no concurrency model, and it re-runs
the entire script on every interaction. It is how you get feedback this week. It is not how you
ship to customers.

---

### For your final project

The pattern in Block 1 works on any document set you control: your own PDFs, your own tickets,
your own notes. Swap the `kb` list for your documents and the rest of the notebook runs
unchanged. If you can state what your project retrieves over, you can have a working prototype
before Week 10.

---

## Readings, mapped to the cells

Every block above corresponds to a specific part of the Week 8 reading. If a block did not make
sense, this is the section to go back to.

| block | steps | what you built | read this |
|---|---|---|---|
| **1** | 1–3 | Chunk, embed, search, prompt, cite | **J&M ch. 11.1–11.3** (sparse and dense retrieval, the bi-encoder) and **11.4** (retrieval-augmented generation) · **HOLLM ch. 8** (semantic search and RAG) · **Tunstall ch. 7** (the retriever-reader pipeline, dense passage retrieval, reranking) |
| **2** | 4 | Greedy against sampling | **Tunstall ch. 5** (greedy search, beam search, temperature, top-k and nucleus sampling) |
| **3** | 5–7 | The same pipeline in LangChain | **HOLLM ch. 7** (chains, prompt templates, composing an LLM application) |
| **4** | 8–10 | CLIP over screenshots | **HOLLM ch. 9** (multimodal LLMs, CLIP's contrastive objective, BLIP-2) |
| **5** | 11–13 | LangGraph, and an agent that retries | **HOLLM ch. 7** (agents, ReAct, tool use) · **J&M ch. 11.4** (agentic and iterative RAG) |
| **6** | 14–15 | Streamlit | No textbook chapter. See the Streamlit docs on `st.status`, `st.session_state` and `@st.cache_resource` |

### Where each specific idea comes from

| the idea | where you met it tonight | source |
|---|---|---|
| Bi-encoder: query and document encoded separately, so passages precompute | STEP 2, the `X` matrix | J&M 11.2; HOLLM ch. 8 |
| Chunking on the document's own structure | STEP 1, one chunk per section heading | HOLLM ch. 8 |
| Grounding a generator on retrieved passages | STEP 3, the `SYSTEM` prompt | J&M 11.4; HOLLM ch. 8 |
| Decoding as a selection rule over a distribution | STEP 4 | Tunstall ch. 5 |
| Chat templates, and why a completion model is not a chat model | STEPS 6–7, the `ChatHuggingFace` fix | Tunstall ch. 5; HOLLM ch. 7 |
| Contrastive image-text pretraining | STEP 9, `get_image_features` | HOLLM ch. 9 |
| ReAct: reason, act, observe, repeat | STEPS 12–13, the grade and widen nodes | HOLLM ch. 7 |
| Refusal as a designed behaviour | STEP 13, `NO_ANSWER` | J&M 11.4 |

### Full references

- Jurafsky, D. and Martin, J. H. *Speech and Language Processing*, 3rd edition draft. Chapter 11.
- Alammar, J. and Grootendorst, M. *Hands-On Large Language Models*. O'Reilly. Chapters 7, 8, 9.
- Tunstall, L., von Werra, L. and Wolf, T. *Natural Language Processing with Transformers*.
  O'Reilly. Chapters 5 and 7.

### Models used tonight

| model | what for | size |
|---|---|---|
| `sentence-transformers/all-MiniLM-L6-v2` | encoding chunks and questions, 384 dimensions | ~90 MB |
| `Qwen/Qwen2.5-1.5B-Instruct` | drafting the reply | ~3.1 GB in float16 |
| `openai/clip-vit-base-patch32` | putting screenshots and text in one space | ~600 MB |

None of them were trained or fine-tuned tonight. Every one is used exactly as downloaded,
which is rung 3 of the adaptation ladder from Lecture 14.